# Simple LR + Bagging Ensemble

This notebook combines Logistic Regression and bagged decision trees using the previously selected soft-voting weights: **95% LR + 5% bagging**.

Run the cells in order to load the data, remove duplicate measurements, evaluate the models on a grouped validation split, and generate a submission. Place the CSVs in data/ or the current directory.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import BaggingClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
TARGET = "vehicle_class"
ID = "vehicle_id"


## 1. Load and preprocess

Exclude the record ID, convert nonfinite measurements to missing values, and remove repeated measurement-and-label rows. Imputation is fitted inside each model's pipeline.

In [2]:
data_dir = Path("data") if Path("data/vehicle_train_10000.csv").exists() else Path(".")

train = pd.read_csv(data_dir / "vehicle_train_10000.csv")
test = pd.read_csv(data_dir / "vehicle_test.csv")

features = [c for c in train.columns if c not in [ID, TARGET]]
assert set(test.columns) == set(features + [ID])
assert train[TARGET].notna().all()
assert set(train[TARGET]) == {0, 1, 2}

train[features] = train[features].replace([np.inf, -np.inf], np.nan)
test[features] = test[features].replace([np.inf, -np.inf], np.nan)

train = train.drop_duplicates(features + [TARGET]).reset_index(drop=True)

X = train[features]
y = train[TARGET].astype(int)
X_test = test[features]

print("Cleaned training rows:", len(train))
print("Test rows:", len(test))


Cleaned training rows: 7500
Test rows: 2500


## 2. Define the ensemble

LR uses median imputation and standard scaling. Bagging uses median imputation and 80 decision trees with a minimum leaf size of 5. Soft voting combines their class probabilities with fixed weights.

In [3]:
lr = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=3000, solver="lbfgs"),
)

bagging = make_pipeline(
    SimpleImputer(strategy="median"),
    BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=None,
            min_samples_leaf=5,
            random_state=SEED,
        ),
        n_estimators=80,
        max_samples=0.8,
        max_features=1.0,
        bootstrap=True,
        random_state=SEED,
        n_jobs=1,
    ),
)

ensemble = VotingClassifier(
    estimators=[("lr", lr), ("bagging", bagging)],
    voting="soft",
    weights=[0.95, 0.05],
    n_jobs=1,
)


## 3. Validate

Reserve approximately 20% of the cleaned data. Identical measurements stay in the same split, including any conflicting-label copies. Compare all three predictions on the same vehicles; the blend weights remain fixed.

In [4]:
groups = train.groupby(features, dropna=False, sort=False).ngroup()

splitter = StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=2026
)
train_idx, valid_idx = next(splitter.split(X, y, groups))
assert set(groups.iloc[train_idx]).isdisjoint(groups.iloc[valid_idx])

X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

ensemble.fit(X_train, y_train)

for name, fitted_model in [
    ("LR", ensemble.named_estimators_["lr"]),
    ("Bagging", ensemble.named_estimators_["bagging"]),
    ("Ensemble", ensemble),
]:
    predictions = fitted_model.predict(X_valid)
    print(f"{name} validation accuracy: {accuracy_score(y_valid, predictions):.2%}")

print("\nEnsemble classification report:")
print(classification_report(
    y_valid,
    ensemble.predict(X_valid),
    labels=[0, 1, 2],
    target_names=["Sedan", "SUV", "Sports Car"],
    zero_division=0,
))


LR validation accuracy: 87.73%
Bagging validation accuracy: 87.13%
Ensemble validation accuracy: 87.67%

Ensemble classification report:
              precision    recall  f1-score   support

       Sedan       0.83      0.83      0.83       521
         SUV       0.88      0.91      0.89       495
  Sports Car       0.93      0.90      0.92       484

    accuracy                           0.88      1500
   macro avg       0.88      0.88      0.88      1500
weighted avg       0.88      0.88      0.88      1500



## 4. Train on all cleaned data and generate submission

Refit the same ensemble on every cleaned training row, then predict test rows in their original order. Local validation accuracy is not a competition score.

In [5]:
ensemble.fit(X, y)
test_predictions = ensemble.predict(X_test).astype(int)

submission = pd.DataFrame({
    ID: test[ID].to_numpy(),
    TARGET: test_predictions,
})

assert len(submission) == len(test)
assert submission[ID].is_unique
assert not submission.isna().any().any()
assert submission[TARGET].isin([0, 1, 2]).all()
assert np.array_equal(submission[ID].to_numpy(), test[ID].to_numpy())

submission_path = Path("submission_simple_lr_bagging.csv")
submission.to_csv(submission_path, index=False)

print(submission.head())
print("\nSaved:", submission_path.resolve())


   vehicle_id  vehicle_class
0       10001              0
1       10002              0
2       10003              1
3       10004              2
4       10005              1

Saved: C:\Users\shubh\Downloads\iiitb\academics\ml\project\vehicle\submission_simple_lr_bagging.csv
